# Preliminary Analysis - Hybrid Quantum-Classical Comparison
**Does the Quantum Circuit Actually Contribute? A Controlled Ablation**

---

## Abstract

This notebook is a **preliminary analysis** that isolates the contribution of the quantum
circuit within a simple hybrid model. Five arms share the same data split and the same
8-feature classical head; only the feature extractor (256 -> 8) varies:

| Arm | Extractor | Trainable params | Type |
|-----|-----------|:----------------:|------|
| `trained`  | `hur8` quantum circuit (learned) | 62 | Quantum trainable |
| `frozen`   | `hur8` quantum circuit (random fixed) | 52 | Quantum fixed |
| `mlp8`     | `Linear(256,8) + tanh` (learned) | ~2 108 | Classical trainable |
| `randfeat` | Fixed random projection + cos | 52 | Classical fixed |
| `logistic` | `Linear(256,4)` directly | 1 028 | Classical trivial |

**Same seed = same data split** -> paired comparisons cancel sampling variance.

The notebook is fully self-contained: all circuit definitions are inlined.

## Table of Contents

1. [Environment Setup](#1-environment-setup)
2. [Ablation Design](#2-ablation-design)
3. [Model Architecture](#3-model-architecture)
4. [Running the Experiments](#4-running-the-experiments)
5. [Loading Results](#5-loading-results)
6. [Analysis](#6-analysis)
   - 6.1 Arm Comparison
   - 6.2 Paired Delta Analysis
   - 6.3 Training Dynamics
   - 6.4 Confusion Matrices
   - 6.5 Cost Analysis
7. [Conclusions](#7-conclusions)

## 1. Environment Setup

Install all required packages. The analysis in Sections 5-7 runs entirely on CPU;
GPU is only required for training (Section 4). If the environment has already been
configured for the companion QCNN Benchmark notebook, the cell below is a no-op.

In [ ]:
# Install all required packages (re-run if any import fails).
%pip install -q pennylane==0.45.0 pennylane-lightning==0.45.0 \
              torch torchvision \
              numpy pandas matplotlib seaborn scipy

In [ ]:
import os, sys, json, glob, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from IPython.display import display
warnings.filterwarnings("ignore")

plt.rcParams.update({"figure.dpi": 110, "figure.facecolor": "white",
                     "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.3, "font.size": 11})

ROOT    = Path.cwd()
RESULTS = ROOT / "suitev3" / "results"
SUMMARY = RESULTS / "summary.csv"

CLASS_NAMES = ["T-shirt", "Trouser", "Sneaker", "Bag"]
ARMS = ["trained", "frozen", "mlp8", "randfeat", "logistic"]
ARM_COLORS = {"trained":"#2196F3","frozen":"#03A9F4",
              "mlp8":"#FF9800","randfeat":"#FF5722","logistic":"#9E9E9E"}
ARM_LABELS = {"trained":"Quantum trained","frozen":"Quantum frozen",
              "mlp8":"MLP-8 (classical)","randfeat":"RandFeat (classical)",
              "logistic":"Logistic (trivial)"}

print(f"Results : {RESULTS}")
print(f"summary : {'EXISTS' if SUMMARY.exists() else 'NOT FOUND - see Section 4'}")

## 2. Ablation Design

### 2.1 Research Question

In a hybrid quantum-classical model, **how much does the quantum circuit contribute**
vs. an equally wide classical extractor?  This is non-trivial for two reasons:

1. The quantum model has only **10 trainable parameters** in its convolution layer - far
   fewer than any classical bottleneck of the same width (256 -> 8 requires 2 056 weights).
2. However, *amplitude encoding* of 256 pixel values on 8 qubits requires approximately
   **254 CNOT gates** and ~10 000 T-gates per inference - orders of magnitude more than
   the ~4 200 FLOPs of the classical twin. Counting only trainable parameters is misleading.

### 2.2 The 2 x 2 Factorial

```
                  TRAINABLE          FIXED (random)
              +-----------------+---------------------+
  QUANTUM     |    trained      |       frozen         |
              +-----------------+---------------------+
  CLASSICAL   |     mlp8        |      randfeat        |
              +-----------------+---------------------+
              + logistic  (trivial linear reference)
```

### 2.3 Paired Comparisons

Same seed -> same data split across all arms (no sampling variance in deltas):

| Comparison | What it measures |
|------------|-----------------|
| `trained - mlp8` | Quantum vs classical trainable extractor |
| `frozen - randfeat` | Quantum vs classical random features |
| `trained - frozen` | Value of *learning* the circuit |
| `trained - logistic` | Hybrid model vs trivial linear classifier |

## 3. Model Architecture

### 3.1 HybridFlatQCNN (quantum arms)

```
x in R256
  -> AmplitudeEmbedding(x, wires=0..7)          [fixed encoding circuit]
  -> hur8 conv on all 8 wires  (theta in R10)       [trained or frozen]
  -> <Z>0...<Z>7                                  [8 features in [-1,1]]
  -> LayerNorm(8) + Linear(8,4)                  [52 trainable head params]
  -> logits in R4
```

### 3.2 Classical Baselines

| Arm | Architecture | Trainable params |
|-----|-------------|:----------------:|
| `mlp8` | `Linear(256,8) -> tanh -> LayerNorm(8) -> Linear(8,4)` | ~2 108 |
| `randfeat` | Fixed cos-random proj: `cos(xW+b)` -> head | 52 (head only) |
| `logistic` | `Linear(256,4)` | 1 028 |

`randfeat` is deliberately **iso-parameter** with `frozen` (52 params each).

### 3.3 Frozen-Arm Optimisation

The frozen circuit produces identical features every epoch. The training loop
**precomputes features once** and trains only the head on cached tensors, reducing
~20 quantum passes to **1** (no accuracy change, ~15x speedup).

In [ ]:
import pennylane as qml

# -- Ansatz circuits (inlined from suitev2/ansatz.py) -------------------------

def hur6(theta, wires):
    """SO(4) circuit, 6 parameters."""
    a, b = wires
    qml.RY(theta[0], wires=a); qml.RY(theta[1], wires=b)
    qml.CNOT(wires=[a, b])
    qml.RY(theta[2], wires=a); qml.RY(theta[3], wires=b)
    qml.CNOT(wires=[a, b])
    qml.RY(theta[4], wires=a); qml.RY(theta[5], wires=b)

def hur8(theta, wires):
    """Hur 2022 baseline circuit, 10 parameters."""
    a, b = wires
    qml.RX(theta[0], wires=a); qml.RX(theta[1], wires=b)
    qml.RZ(theta[2], wires=a); qml.RZ(theta[3], wires=b)
    qml.RX(theta[4], wires=a); qml.RX(theta[5], wires=b)
    qml.CNOT(wires=[a, b])
    qml.RX(theta[6], wires=a); qml.RX(theta[7], wires=b)
    qml.RZ(theta[8], wires=a); qml.RZ(theta[9], wires=b)

def hur9(theta, wires):
    """SU(4) / KAK circuit, 15 parameters."""
    a, b = wires
    qml.U3(theta[0], theta[1], theta[2], wires=a)
    qml.U3(theta[3], theta[4], theta[5], wires=b)
    qml.CNOT(wires=[a, b])
    qml.RY(theta[6], wires=a); qml.RZ(theta[7], wires=b)
    qml.CNOT(wires=[b, a]); qml.RY(theta[8], wires=a)
    qml.CNOT(wires=[a, b])
    qml.U3(theta[9],  theta[10], theta[11], wires=a)
    qml.U3(theta[12], theta[13], theta[14], wires=b)

def custom_ansatz(theta, wires):
    """Cartan-inspired circuit, 11 parameters."""
    a, b = wires
    qml.RZ(theta[0], wires=a); qml.RY(theta[1], wires=a)
    qml.RZ(theta[2], wires=b); qml.RY(theta[3], wires=b)
    qml.IsingXX(theta[4], wires=[a, b])
    qml.IsingYY(theta[5], wires=[a, b])
    qml.IsingZZ(theta[6], wires=[a, b])
    qml.RY(theta[7], wires=a); qml.RZ(theta[8], wires=a)
    qml.RY(theta[9], wires=b); qml.RZ(theta[10], wires=b)

def conv_layer(conv_fn, theta, active_wires):
    """Translationally invariant conv: even + shifted pairs, shared params."""
    n = len(active_wires)
    even    = [(active_wires[i], active_wires[i+1]) for i in range(0, n-1, 2)]
    shifted = [(active_wires[i], active_wires[i+1]) for i in range(1, n-1, 2)]
    for pair in even + shifted:
        conv_fn(theta, pair)

print("Ansatz circuits defined.")

In [ ]:
# -- Visualise the HybridFlatQCNN quantum backbone ----------------------------
import torch
N = 8
dev = qml.device("default.qubit", wires=N)

@qml.qnode(dev)
def hybrid_circuit(x, theta):
    qml.AmplitudeEmbedding(x, wires=range(N), normalize=True)
    conv_layer(hur8, theta, list(range(N)))
    return [qml.expval(qml.PauliZ(w)) for w in range(N)]

x_d = torch.rand(2**N); x_d /= x_d.norm()
theta_d = torch.zeros(10)
print("HybridFlatQCNN backbone  (AmplitudeEmbedding + hur8 flat conv, 8 <Z> readouts):")
print(qml.draw(hybrid_circuit)(x_d.numpy(), theta_d.numpy()))

## 4. Running the Experiments

> **Skip if `suitev3/results/summary.csv` already exists.**

Run from the **repository root**:

```bash
# Full suite - 5 arms x 3 seeds
python -m suitev3.run_suite

# Only the core quantum/classical pair
python -m suitev3.run_suite --arms trained mlp8

# Smoke test (2 epochs, 40 samples/class, ~5 min)
python -m suitev3.run_suite --epochs 2 --train-per-class 40
```

**Estimated runtimes** (GPU): `trained` ~5-10 min / `frozen` ~2 min (feature caching) /
`mlp8`/`randfeat`/`logistic` < 1 min each / **Full suite ~20-30 min**.

## 5. Loading Results

In [ ]:
df3_raw = None
df3     = None

if not SUMMARY.exists():
    print("WARNING:  summary.csv not found - see Section 4.")
else:
    df3_raw = pd.read_csv(SUMMARY)
    if df3_raw.empty:
        print("WARNING:  summary.csv is empty.")
    else:
        df3 = df3_raw.copy()
        for c in ["test_acc","val_acc_best","test_loss","best_val_loss",
                  "n_params","best_epoch","n_epochs_trained"]:
            if c in df3.columns:
                df3[c] = pd.to_numeric(df3[c], errors="coerce")
        print(f"[OK]  {len(df3)} rows | arms: {sorted(df3['arm'].unique())} | "
              f"seeds: {sorted(df3['seed'].unique())}")
        display(df3[["arm","seed","n_params","n_epochs_trained",
                     "test_acc","val_acc_best","best_epoch"]].sort_values(
                     ["arm","seed"]).to_string(index=False))

In [ ]:
curves3 = {}
if RESULTS.exists():
    for p in sorted(RESULTS.glob("*/metrics.csv")):
        run = p.parent.name
        parts = run.rsplit("_seed", 1)
        if len(parts) == 2:
            key = parts[0]
            try:
                c = pd.read_csv(p)
                for col in ["train_loss","train_acc","val_loss","val_acc"]:
                    if col in c.columns:
                        c[col] = pd.to_numeric(c[col], errors="coerce")
                curves3.setdefault(key, []).append(c)
            except Exception:
                pass
print(f"Curves loaded: {sum(len(v) for v in curves3.values())} runs, "
      f"arms: {sorted(curves3.keys())}")

## 6. Analysis

### 6.1 Arm Comparison

Box plot of test accuracy distribution over the 3 seeds for each arm (box shows median
and interquartile range; dots show individual seed results). Colour distinguishes
quantum arms (blue) from classical arms (orange/red) and the trivial baseline (grey).

In [ ]:
def _nd3(s): print(f"WARNING:  No data for {s} - run Section 4 then reload Section 5.")

if df3 is None:
    _nd3("6.1")
else:
    grp = df3.groupby("arm")["test_acc"].agg(["mean","std","count"]).reset_index()
    grp["std"] = grp["std"].fillna(0)
    ord_idx = {a: i for i, a in enumerate(ARMS)}
    grp["_order"] = grp["arm"].map(ord_idx).fillna(99)
    grp = grp.sort_values("_order").drop(columns=["_order"]).reset_index(drop=True)

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.bar(range(len(grp)), grp["mean"], yerr=grp["std"], capsize=6,
           color=[ARM_COLORS.get(a,"#9E9E9E") for a in grp["arm"]],
           edgecolor="white", alpha=0.9, width=0.6)
    ax.axhline(0.25, ls="--", color="gray", lw=1.2, label="Random chance")
    for i, arm in enumerate(grp["arm"]):
        pts = df3[df3["arm"]==arm]["test_acc"].values
        ax.scatter([i]*len(pts), pts, color="black", s=30, zorder=5, alpha=0.7)
    ax.set_xticks(range(len(grp)))
    ax.set_xticklabels([ARM_LABELS.get(a,a) for a in grp["arm"]], fontsize=10)
    ax.set_ylabel("Mean Test Accuracy"); ax.set_ylim(0, 1.1)
    ax.set_title("Test Accuracy by Arm - Mean +/- Std (dots = individual seeds)", fontsize=13)
    ax.legend(fontsize=9)
    for i, r in grp.iterrows():
        ax.text(i, r["mean"]+r["std"]+0.015, f'{r["mean"]:.3f}', ha="center", fontsize=9)
    plt.tight_layout(); plt.show()
    display(grp.rename(columns={"mean":"mean_acc","std":"std_acc","count":"n_seeds"}).round(4).to_string(index=False))

### 6.2 Paired Delta Analysis

Within-seed accuracy deltas (same seed = same images). Eliminates data-sampling variance.
Positive Delta = first arm better; dots = per-seed deltas.

In [ ]:
if df3 is None:
    _nd3("6.2")
else:
    pivot3 = df3.pivot_table(index="seed", columns="arm", values="test_acc")
    seeds  = pivot3.index.tolist()

    COMPS = [
        ("trained","mlp8",    "Quantum trained\nvs MLP-8"),
        ("frozen", "randfeat","Quantum frozen\nvs RandFeat"),
        ("trained","frozen",  "Trained\nvs Frozen"),
        ("trained","logistic","Trained\nvs Logistic"),
    ]
    valid = [(a,b,l) for a,b,l in COMPS if a in pivot3.columns and b in pivot3.columns]

    if not valid:
        print("WARNING:  Not enough arms for paired deltas. Need >=2 arms completed.")
    else:
        fig, axes = plt.subplots(1, len(valid), figsize=(5*len(valid), 4.5))
        if len(valid) == 1: axes = [axes]
        for ax, (a1,a2,lbl) in zip(axes, valid):
            d = pivot3[a1] - pivot3[a2]
            mean_d, std_d = d.mean(), d.std()
            color = "#2196F3" if mean_d >= 0 else "#F44336"
            ax.bar([0], [mean_d], color=color, alpha=0.7, width=0.4,
                   yerr=std_d, capsize=8, edgecolor="white")
            for s, v in zip(seeds, d):
                ax.scatter(0, v, color="black", s=40, zorder=5)
                ax.annotate(f"s{s}", (0,v), textcoords="offset points",
                            xytext=(10,0), fontsize=8)
            ax.axhline(0, color="black", lw=1)
            ax.set_xlim(-0.5, 0.5); ax.set_xticks([])
            ax.set_ylabel(f"Delta ({a1} - {a2})")
            ax.set_title(lbl, fontsize=9)
            ax.text(0, mean_d+(0.02 if mean_d>=0 else -0.05),
                    f'Delta={mean_d:+.3f}\n+/-{std_d:.3f}',
                    ha="center", fontsize=9, fontweight="bold")
        fig.suptitle("Paired Delta Accuracy (same seed = same data split)", fontsize=13, y=1.02)
        plt.tight_layout(); plt.show()

        print("\nPaired delta summary:")
        print(f"  {'Comparison':<25} {'mean Delta':>8} {'std Delta':>7}  per-seed")
        for a1,a2,_ in valid:
            if a1 in pivot3.columns and a2 in pivot3.columns:
                d = pivot3[a1] - pivot3[a2]
                print(f"  {a1+' - '+a2:<25} {d.mean():>+8.4f} {d.std():>7.4f}  "
                      f"{[f'{v:+.4f}' for v in d.values]}")

### 6.3 Training Dynamics

Train accuracy (solid) and validation accuracy (dashed) for each arm, mean +/- std over seeds.
Gap between curves indicates overfitting.

In [ ]:
if not curves3:
    _nd3("6.3 - no metrics.csv found")
else:
    fig, ax = plt.subplots(figsize=(9, 5))
    for arm, seed_dfs in sorted(curves3.items()):
        max_ep = max(c["epoch"].max() for c in seed_dfs)
        tr_mat = np.full((len(seed_dfs), int(max_ep)), np.nan)
        va_mat = np.full((len(seed_dfs), int(max_ep)), np.nan)
        for si, c in enumerate(seed_dfs):
            ep = c["epoch"].astype(int).values - 1
            for e, ta, va in zip(ep, c["train_acc"].values, c["val_acc"].values):
                if 0 <= e < int(max_ep):
                    tr_mat[si,e]=ta; va_mat[si,e]=va
        x = np.arange(1, int(max_ep)+1)
        color = ARM_COLORS.get(arm,"#9E9E9E")
        label = ARM_LABELS.get(arm,arm)
        m_tr = np.nanmean(tr_mat, 0); s_tr = np.nanstd(tr_mat, 0)
        m_va = np.nanmean(va_mat, 0); s_va = np.nanstd(va_mat, 0)
        ax.plot(x, m_tr, color=color, lw=2, label=f"{label} (train)")
        ax.fill_between(x, m_tr-s_tr, m_tr+s_tr, alpha=0.10, color=color)
        ax.plot(x, m_va, color=color, lw=2, ls="--")
        ax.fill_between(x, m_va-s_va, m_va+s_va, alpha=0.07, color=color)
    ax.axhline(0.25, ls=":", color="gray", lw=0.8, label="Random chance")
    ax.set_xlabel("Epoch"); ax.set_ylabel("Accuracy"); ax.set_ylim(0,1.05)
    ax.set_title("Train (solid) vs Val (dashed) Accuracy per Arm", fontsize=13)
    ax.legend(fontsize=8, loc="lower right", ncol=2)
    plt.tight_layout(); plt.show()

### 6.4 Confusion Matrices

One per arm, summed over seeds.

In [ ]:
def _extract_cm(row, n=4):
    cm = np.zeros((n,n), dtype=int)
    for i in range(n):
        for j in range(n):
            col=f"cm_{i}{j}"
            if col in row.index and not pd.isna(row[col]):
                cm[i,j]=int(row[col])
    return cm

if df3 is None:
    _nd3("6.4")
else:
    present = [a for a in ARMS if a in df3["arm"].unique()]
    ncols = min(3, len(present)); nrows = (len(present)+ncols-1)//ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(5*ncols,4.5*nrows), squeeze=False)
    axf = axes.flatten()
    for idx, arm in enumerate(present):
        sub = df3[df3["arm"]==arm]
        cm = sum(_extract_cm(r) for _,r in sub.iterrows())
        norm = cm / cm.sum(axis=1, keepdims=True).clip(min=1)
        sns.heatmap(norm, annot=cm, fmt="d", cmap="Blues", vmin=0, vmax=1,
                    xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
                    ax=axf[idx], cbar=False, annot_kws={"size":9})
        axf[idx].set_title(f"{ARM_LABELS.get(arm,arm)}  (n={len(sub)})", fontsize=9)
        axf[idx].set_xlabel("Predicted", fontsize=8); axf[idx].set_ylabel("True", fontsize=8)
    for ax in axf[len(present):]: ax.set_visible(False)
    fig.suptitle("Confusion Matrices - summed over seeds", fontsize=13, y=1.01)
    plt.tight_layout(); plt.show()

### 6.5 Cost Analysis

A fair cost comparison must include the **data encoding circuit**, not just trainable
parameters. Amplitude embedding of 256 values on 8 qubits requires approximately 254 CNOT
gates, which represents **97 % of the total two-qubit gate budget** of the entire quantum
model. The 10 trainable `hur8` parameters add only approximately 7 additional CNOTs.

Using the Ross-Selinger approximation (approximately 3 x log2(1/eps) T-gates per
single-qubit rotation, with synthesis precision eps = 1e-3), the full quantum model
requires approximately **10 000 T-gates per inference**. By contrast, the MLP-8 classical
twin requires approximately 4 200 floating-point operations per inference.

In [ ]:
# Static cost figures derived from suitev3/cost.py analysis
COST = {
    "Arm":              ["Quantum trained/frozen", "mlp8",   "randfeat", "logistic"],
    "Trainable params": [62,                        "~2 108", 52,         1028],
    "CNOT gates":       ["~261 (enc: ~254, conv: ~7)", "-",  "-",        "-"],
    "T-gates (eps=1e-3)": ["~10 000",                "-",     "-",        "-"],
    "FLOPs / inference":["- (circuit)",            "~4 200", "~4 200",   "~2 048"],
}
print("Cost comparison:")
display(pd.DataFrame(COST).to_string(index=False))
print()
print("Key takeaway: the '10 trainable parameters' figure omits the encoding circuit.")
print("Counting the full quantum model: ~10^4 T-gates vs ~10^3 FLOPs for the classical twin.")

fig, axes = plt.subplots(1,2, figsize=(11,4))

arms_p = ["trained","frozen","mlp8","randfeat","logistic"]
params_ = [62, 52, 2108, 52, 1028]
axes[0].barh(range(5), params_, color=[ARM_COLORS[a] for a in arms_p], alpha=0.85)
axes[0].set_yticks(range(5)); axes[0].set_yticklabels([ARM_LABELS[a] for a in arms_p], fontsize=9)
axes[0].set_xlabel("Trainable Parameters"); axes[0].set_title("Trainable Parameters", fontsize=11)
for i,v in enumerate(params_): axes[0].text(v+30, i, str(v), va="center", fontsize=9)

cl_names = ["mlp8","randfeat","logistic"]; cl_flops = [4200,4200,2048]
axes[1].bar([0,1,2], cl_flops, color=[ARM_COLORS[a] for a in cl_names], alpha=0.85)
axes[1].bar([3], [10000], color=ARM_COLORS["trained"], alpha=0.85)
axes[1].set_yscale("log")
axes[1].set_xticks([0,1,2,3])
axes[1].set_xticklabels(["mlp8\n(FLOPs)","randfeat\n(FLOPs)","logistic\n(FLOPs)","quantum\n(T-gates)"], fontsize=9)
axes[1].set_ylabel("Operations (log scale)"); axes[1].set_title("Inference Cost", fontsize=11)
for i,v in enumerate(cl_flops): axes[1].text(i, v*1.3, f"{v:,}", ha="center", fontsize=8)
axes[1].text(3, 10000*1.3, "~10 000", ha="center", fontsize=8)
fig.suptitle("Resource Cost: Parameters and Inference Operations", fontsize=12, y=1.01)
plt.tight_layout(); plt.show()

## 7. Conclusions

### 7.1 Reading the Paired Deltas

| Comparison | Interpretation |
|------------|----------------|
| `trained - mlp8 > 0` | Quantum trainable extractor outperforms iso-architecture classical twin |
| `trained - mlp8 ~ 0` | No accuracy advantage for quantum over classical (at this scale) |
| `frozen - randfeat > 0` | Quantum random features more informative than classical random projections |
| `trained - frozen > 0` | Training the quantum circuit adds value over random initialisation |
| `trained - logistic > 0` | Hybrid model meaningfully outperforms the trivial linear classifier |

### 7.2 Encoding Cost Argument

Even if `trained >= mlp8` in accuracy:
1. The quantum model incurs **~10 000 T-gates** per inference (encoding-dominated) vs
   **~4 200 FLOPs** for `mlp8` - a ~10x disadvantage in hardware cost.
2. The `frozen` arm is iso-parameter with `randfeat` (52 trainable params each), making
   that the cleanest quantum vs. classical comparison.
3. The "10 trainable parameters" is not a valid cost proxy: 97 % of the quantum circuit's
   CNOT gates come from the fixed amplitude-encoding state preparation.

### 7.3 Relationship to the Main QCNN Benchmark

The QCNN Benchmark notebook (see companion notebook) explores the full two-layer QCNN
across multiple ansatz and encodings. This preliminary analysis asks the complementary
question: in a *simpler* single-layer hybrid model, is the quantum circuit necessary at all?
The two studies address orthogonal questions and should be read together.

### References
- Schuld, M. (2021). *Supervised quantum ML models are kernel methods.* arXiv:2101.11020.
- Havlicek, V. et al. (2019). *Supervised learning with quantum-enhanced feature spaces.* Nature.
- Ross, N. J. & Selinger, P. (2016). *Optimal ancilla-free Clifford+T approximation.* QIC.
- Hur, T., Kim, L. & Park, D. K. (2022). *QCNN for classical data classification.* QMI.